# 02 · Latent state geometry

Encode the cohort into the six-dimensional latent state space, project to 2D, and look at the cloud.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt


In [ ]:
from data.synthetic_generator import generate_synthetic_cohort
from features import engineer_all_features
from states.latent_state_encoder import encode_latent_states_classical, LATENT_DIM_NAMES
from states.state_geometry import project_latent_2d

cohort = engineer_all_features(generate_synthetic_cohort(n_participants=80, n_days=120, seed=17))


In [ ]:
def pick(df, cols):
    present = [c for c in cols if c in df.columns]
    return df[present].to_numpy(dtype=float, na_value=0.0) if present else np.zeros((len(df), len(cols)))

W = pick(cohort, ['sleep_duration_hours','hrv_rmssd','resting_hr','daily_steps','recovery_score'])
B = pick(cohort, ['screen_time_minutes','mobility_radius_km','location_entropy','phone_unlock_count'])
C = pick(cohort, ['temperature_c','nighttime_temperature_c','aqi','heat_wave_flag'])
M = pick(cohort, ['missing_wearable_flag','missing_phone_flag','missing_survey_flag'])
P = pick(cohort, ['baseline_hrv','baseline_resting_hr','baseline_climate_vulnerability','baseline_resilience'])
Z = encode_latent_states_classical(W, B, C, M, P).latent
Z.shape, LATENT_DIM_NAMES


## 2D projection


In [ ]:
Z2 = project_latent_2d(Z)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(Z2[:,0], Z2[:,1], s=6, alpha=0.4)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('Latent state cloud')
plt.tight_layout(); plt.show()
